# Task 3 — Domain Generalization (PACS)
Reuses Task 2's ERM checkpoint unchanged and the same source-domain pipeline.
**No Sketch (target) data is loaded anywhere except the final evaluation section** —
per spec, no target image may influence Task 3 training, checkpoint selection, or
hyperparameter decisions. Structured for idempotent re-runs: run top to bottom;
if the session restarts, Run All just works.


## 0. Setup — imports, seed, BatchNorm-freeze utility

In [32]:
import os
import copy
import random
import numpy as np

import torch
import torch.nn as nn
from torch.optim import AdamW
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset, ConcatDataset

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def set_seed(seed=6304):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(6304)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def freeze_bn_stats(m):
    """Freezes BatchNorm running statistics at their pretrained values but
    keeps scale (gamma) and bias (beta) trainable. Call model.apply(freeze_bn_stats)
    AFTER model.train() every epoch, for every method — same shared policy as Task 2,
    including both SAM forward/backward passes."""
    if m.__class__.__name__.find('BatchNorm') != -1:
        m.eval()

print("Setup complete.")

Using device: cuda
Setup complete.


## 1. Data — source-only loaders (Photo / Art Painting / Cartoon)
**No Sketch loader here.** Same stratified 80/20 splits, seed 6304, as Task 2, so
results are directly comparable. Sketch is loaded only in the final evaluation
section, after every Task 3 decision is frozen.

In [34]:
base_dir = ""
for root, dirs, files in os.walk('/kaggle/input/'):
    if {'photo', 'art_painting', 'cartoon', 'sketch'}.issubset({d.lower() for d in dirs}):
        base_dir = root
        break

print(f"Dataset located at: {base_dir}")

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def get_stratified_split(dataset, seed=6304, test_size=0.2):
    labels = dataset.targets
    train_idx, val_idx = train_test_split(
        range(len(labels)), test_size=test_size, stratify=labels, random_state=seed
    )
    return train_idx, val_idx

sources = ['photo', 'art_painting', 'cartoon']
source_loaders = {}

for domain in sources:
    domain_path = os.path.join(base_dir, domain)
    full_train_ds = ImageFolder(root=domain_path, transform=train_transform)
    full_val_ds = ImageFolder(root=domain_path, transform=eval_transform)

    train_idx, val_idx = get_stratified_split(full_train_ds, seed=6304)

    train_subset = Subset(full_train_ds, train_idx)
    val_subset = Subset(full_val_ds, val_idx)

    source_loaders[domain] = {
        'train': DataLoader(train_subset, batch_size=8, shuffle=True, drop_last=True),
        'val': DataLoader(val_subset, batch_size=32, shuffle=False)
    }

print("Source loaders ready (no target/Sketch data loaded):")
for d in sources:
    print(f"  {d}: {len(source_loaders[d]['train'].dataset)} train / {len(source_loaders[d]['val'].dataset)} val")

Dataset located at: /kaggle/input/datasets/nickfratto/pacs-dataset/pacs_data/pacs_data
Source loaders ready (no target/Sketch data loaded):
  photo: 1336 train / 334 val
  art_painting: 1638 train / 410 val
  cartoon: 1875 train / 469 val


## 2. Shared evaluation utility

In [35]:
def evaluate_loader(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return acc * 100, macro_f1 * 100

def report_source(model, tag):
    print(f"--- {tag}: Source Validation ---")
    f1s = []
    for domain in ['photo', 'art_painting', 'cartoon']:
        acc, f1 = evaluate_loader(model, source_loaders[domain]['val'])
        f1s.append(f1)
        print(f"  [{domain.capitalize()}]: Accuracy = {acc:.2f}% | Macro-F1 = {f1:.2f}%")
    mean_f1 = np.mean(f1s)
    worst_f1 = np.min(f1s)
    print(f"  Mean Source F1: {mean_f1:.2f}% | Worst-Source F1: {worst_f1:.2f}%")
    return mean_f1, worst_f1


## 3. ERM baseline (reused from Task 2, unchanged)
Load-only — do **not** retrain. Per spec this is the same checkpoint used as the
Task 2 Source-only model. Update the path below to match your Kaggle Dataset name.

In [36]:
# Adjust this path to match the Kaggle Dataset you created from source_only_erm.pth
ERM_CHECKPOINT_PATH = '/kaggle/input/datasets/ramizabbas/pacs-erm-checkpoint/source_only_erm.pth'
# /kaggle/input/datasets/ramizabbas/pacs-erm-checkpoint/source_only_erm.pth

erm_model = models.resnet18(weights=None)
erm_model.fc = nn.Linear(erm_model.fc.in_features, 7)
erm_model.load_state_dict(torch.load(ERM_CHECKPOINT_PATH))
erm_model.to(device)

erm_mean_f1, erm_worst_f1 = report_source(erm_model, "ERM (Task 2 checkpoint, reused)")

--- ERM (Task 2 checkpoint, reused): Source Validation ---
  [Photo]: Accuracy = 96.41% | Macro-F1 = 95.82%
  [Art_painting]: Accuracy = 92.20% | Macro-F1 = 91.72%
  [Cartoon]: Accuracy = 94.46% | Macro-F1 = 94.43%
  Mean Source F1: 93.99% | Worst-Source F1: 91.72%


## 4. DAN-DG — Pairwise Source-Domain Alignment
### Architecture + MMD utility
Same feature-extractor/classifier structure and multi-kernel MMD as Task 2's DAN —
but here the MMD penalty is applied **pairwise across the three source domains only**
(Photo-Art, Photo-Cartoon, Art-Cartoon), never against target. No adversarial
component, so this should train as stably as Task 2's DAN did.


In [21]:
class DANDG_ResNet(nn.Module):
    def __init__(self, num_classes=7):
        super(DANDG_ResNet, self).__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
        in_features = resnet.fc.in_features
        self.class_classifier = nn.Linear(in_features, num_classes)

    def forward(self, x, return_features=False):
        features = self.feature_extractor(x)
        features = features.view(features.size(0), -1)
        class_preds = self.class_classifier(features)
        if return_features:
            return class_preds, features
        return class_preds


def compute_mmd(feats_a, feats_b):
    """Multi-kernel MMD^2 between two feature sets, using a sum of three RBF
    kernels with bandwidths 0.5x, 1x, 2x the median pairwise squared distance
    of the combined batch, per spec."""
    combined = torch.cat([feats_a, feats_b], dim=0)
    n = combined.size(0)
    dist_sq = torch.cdist(combined, combined, p=2) ** 2

    off_diag_mask = ~torch.eye(n, dtype=torch.bool, device=combined.device)
    median_sq_dist = dist_sq[off_diag_mask].median()
    if median_sq_dist <= 0:
        median_sq_dist = torch.tensor(1.0, device=combined.device)

    bandwidths = [0.5 * median_sq_dist, 1.0 * median_sq_dist, 2.0 * median_sq_dist]
    K = sum(torch.exp(-dist_sq / bw) for bw in bandwidths)

    n_a = feats_a.size(0)
    K_aa = K[:n_a, :n_a]
    K_bb = K[n_a:, n_a:]
    K_ab = K[:n_a, n_a:]

    return K_aa.mean() + K_bb.mean() - 2 * K_ab.mean()

print("DAN-DG architecture and MMD utility defined.")


DAN-DG architecture and MMD utility defined.


### Training
lambda_dg = 1 for the main comparison, per spec. Same AdamW/seed/checkpoint-selection
pattern as Task 2.


In [ ]:
set_seed(6304)

dandg_model = DANDG_ResNet()
dandg_model.to(device)

optimizer = AdamW(dandg_model.parameters(), lr=1e-4, weight_decay=1e-4, eps=1e-4)
class_criterion = nn.CrossEntropyLoss()
lambda_dg = 1.0

max_epochs = 30
patience = 5
best_mean_f1 = -1
epochs_no_improve = 0
best_state = None

photo_loader = source_loaders['photo']['train']
art_loader = source_loaders['art_painting']['train']
cartoon_loader = source_loaders['cartoon']['train']

print("Starting DAN-DG Training...")

for epoch in range(max_epochs):
    dandg_model.train()
    dandg_model.apply(freeze_bn_stats)

    total_class_loss, total_mmd_loss = 0, 0

    for batch_idx, ((img_p, lbl_p), (img_a, lbl_a), (img_c, lbl_c)) in enumerate(
        zip(photo_loader, art_loader, cartoon_loader)):

        img_p, lbl_p = img_p.to(device), lbl_p.to(device)
        img_a, lbl_a = img_a.to(device), lbl_a.to(device)
        img_c, lbl_c = img_c.to(device), lbl_c.to(device)

        preds_p, feats_p = dandg_model(img_p, return_features=True)
        preds_a, feats_a = dandg_model(img_a, return_features=True)
        preds_c, feats_c = dandg_model(img_c, return_features=True)

        all_preds = torch.cat([preds_p, preds_a, preds_c], dim=0)
        all_labels = torch.cat([lbl_p, lbl_a, lbl_c], dim=0)
        loss_class = class_criterion(all_preds, all_labels)

        mmd_pa = compute_mmd(feats_p, feats_a)
        mmd_pc = compute_mmd(feats_p, feats_c)
        mmd_ac = compute_mmd(feats_a, feats_c)
        loss_mmd = (mmd_pa + mmd_pc + mmd_ac) / 3.0

        total_loss = loss_class + lambda_dg * loss_mmd

        optimizer.zero_grad()
        total_loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(dandg_model.parameters(), max_norm=1.0)

        if not torch.isfinite(grad_norm):
            optimizer.zero_grad()
            if batch_idx % 20 == 0:
                print(f"  batch {batch_idx}: SKIPPED (non-finite grad)")
            continue

        optimizer.step()

        total_class_loss += loss_class.item()
        total_mmd_loss += loss_mmd.item()

        if batch_idx % 20 == 0:
            print(f"  batch {batch_idx}: class={loss_class.item():.3f}, "
                  f"mmd={loss_mmd.item():.4f}, grad_norm={grad_norm.item():.3f}")

    dandg_model.eval()
    f1s = [evaluate_loader(dandg_model, source_loaders[d]['val'])[1] for d in ['photo', 'art_painting', 'cartoon']]
    mean_f1 = np.mean(f1s)

    print(f"Epoch {epoch+1}/{max_epochs} | Class Loss: {total_class_loss:.4f} | "
          f"MMD Loss: {total_mmd_loss:.4f} | Mean Source Val F1: {mean_f1:.2f}%")

    if mean_f1 > best_mean_f1:
        best_mean_f1 = mean_f1
        best_state = copy.deepcopy(dandg_model.state_dict())
        epochs_no_improve = 0
        torch.save(best_state, '/kaggle/working/dandg_model.pth')
        print(f"  → new best (F1={mean_f1:.2f}%), checkpoint saved.")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1} (best mean F1: {best_mean_f1:.2f}%)")
            break

print(f"DAN-DG training complete. Best checkpoint (mean source F1={best_mean_f1:.2f}%) saved.")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 174MB/s]


Starting DAN-DG Training...
  batch 0: class=2.119, mmd=0.7399, grad_norm=13.091
  batch 20: class=0.600, mmd=0.7202, grad_norm=17.535
  batch 40: class=0.717, mmd=0.5738, grad_norm=28.242
  batch 60: class=0.388, mmd=0.5278, grad_norm=15.702
  batch 80: class=0.551, mmd=0.4740, grad_norm=14.644
  batch 100: class=0.242, mmd=0.4080, grad_norm=10.929
  batch 120: class=0.204, mmd=0.7507, grad_norm=12.928
  batch 140: class=0.444, mmd=0.5362, grad_norm=16.184
  batch 160: class=0.280, mmd=0.4447, grad_norm=18.330
Epoch 1/30 | Class Loss: 95.0751 | MMD Loss: 89.5210 | Mean Source Val F1: 85.83%
  → new best (F1=85.83%), checkpoint saved.
  batch 0: class=0.392, mmd=0.4132, grad_norm=14.723
  batch 20: class=0.164, mmd=0.4926, grad_norm=7.210
  batch 40: class=0.103, mmd=0.8134, grad_norm=14.124
  batch 60: class=0.326, mmd=0.4575, grad_norm=15.205
  batch 80: class=0.270, mmd=0.4957, grad_norm=10.295
  batch 100: class=0.263, mmd=0.4098, grad_norm=12.299
  batch 120: class=0.501, mmd=0.63

In [37]:
# --- DAN-DG evaluation ---
dandg_model = DANDG_ResNet()
dandg_model.load_state_dict(torch.load('/kaggle/working/dandg_model.pth'))
dandg_model.to(device)
dandg_mean_f1, dandg_worst_f1 = report_source(dandg_model, "DAN-DG")


--- DAN-DG: Source Validation ---
  [Photo]: Accuracy = 97.01% | Macro-F1 = 96.54%
  [Art_painting]: Accuracy = 91.95% | Macro-F1 = 91.94%
  [Cartoon]: Accuracy = 94.88% | Macro-F1 = 95.21%
  Mean Source F1: 94.56% | Worst-Source F1: 91.94%


## 5. SAM — Parameter-Space Stability
### Architecture
Plain ResNet-18 classifier — identical to ERM's architecture. SAM changes the
*optimization procedure*, not the model.


In [38]:
def build_sam_model(num_classes=7):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

print("SAM uses the plain ResNet-18 architecture (same as ERM).")


SAM uses the plain ResNet-18 architecture (same as ERM).


### Training
Standard non-adaptive SAM, rho=0.05. Each step: compute gradient at current weights,
take a normalized ascent step of radius rho, recompute gradient at the perturbed
point, restore original weights, then apply the optimizer update using the
perturbed-point gradient. Two forward/backward passes per batch. The shared frozen-
BatchNorm policy applies during both passes.


In [39]:
set_seed(6304)

sam_model = build_sam_model()
sam_model.to(device)

optimizer = AdamW(sam_model.parameters(), lr=1e-4, weight_decay=1e-4)
class_criterion = nn.CrossEntropyLoss()
rho = 0.05

max_epochs = 30
patience = 5
best_mean_f1 = -1
epochs_no_improve = 0
best_state = None

photo_loader = source_loaders['photo']['train']
art_loader = source_loaders['art_painting']['train']
cartoon_loader = source_loaders['cartoon']['train']

print("Starting SAM Training...")

for epoch in range(max_epochs):
    sam_model.train()
    sam_model.apply(freeze_bn_stats)

    total_loss_sum = 0

    for batch_idx, ((img_p, lbl_p), (img_a, lbl_a), (img_c, lbl_c)) in enumerate(
        zip(photo_loader, art_loader, cartoon_loader)):

        images = torch.cat([img_p, img_a, img_c], dim=0).to(device)
        labels = torch.cat([lbl_p, lbl_a, lbl_c], dim=0).to(device)

        # --- first pass: compute gradient at current weights ---
        optimizer.zero_grad()
        outputs = sam_model(images)
        loss = class_criterion(outputs, labels)
        loss.backward()

        params_with_grad = [p for p in sam_model.parameters() if p.grad is not None]
        grad_norm = torch.norm(torch.stack([p.grad.norm(p=2) for p in params_with_grad]), p=2)

        if not torch.isfinite(grad_norm):
            optimizer.zero_grad()
            if batch_idx % 20 == 0:
                print(f"  batch {batch_idx}: SKIPPED (non-finite grad on ascent step)")
            continue

        scale = rho / (grad_norm + 1e-12)
        e_ws = []
        with torch.no_grad():
            for p in params_with_grad:
                e_w = p.grad * scale
                p.add_(e_w)
                e_ws.append(e_w)

        # --- second pass: gradient at the perturbed point ---
        optimizer.zero_grad()
        outputs2 = sam_model(images)
        loss2 = class_criterion(outputs2, labels)
        loss2.backward()

        # restore original weights before stepping
        with torch.no_grad():
            for p, e_w in zip(params_with_grad, e_ws):
                p.sub_(e_w)

        grad_norm2 = torch.norm(torch.stack([p.grad.norm(p=2) for p in params_with_grad if p.grad is not None]), p=2)
        if not torch.isfinite(grad_norm2):
            optimizer.zero_grad()
            if batch_idx % 20 == 0:
                print(f"  batch {batch_idx}: SKIPPED (non-finite grad on descent step)")
            continue

        optimizer.step()

        total_loss_sum += loss.item()

        if batch_idx % 20 == 0:
            print(f"  batch {batch_idx}: loss={loss.item():.3f}, "
                  f"ascent_grad_norm={grad_norm.item():.3f}")

    sam_model.eval()
    f1s = [evaluate_loader(sam_model, source_loaders[d]['val'])[1] for d in ['photo', 'art_painting', 'cartoon']]
    mean_f1 = np.mean(f1s)

    print(f"Epoch {epoch+1}/{max_epochs} | Loss: {total_loss_sum:.4f} | Mean Source Val F1: {mean_f1:.2f}%")

    if mean_f1 > best_mean_f1:
        best_mean_f1 = mean_f1
        best_state = copy.deepcopy(sam_model.state_dict())
        epochs_no_improve = 0
        torch.save(best_state, '/kaggle/working/sam_model.pth')
        print(f"  → new best (F1={mean_f1:.2f}%), checkpoint saved.")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1} (best mean F1: {best_mean_f1:.2f}%)")
            break

print(f"SAM training complete. Best checkpoint (mean source F1={best_mean_f1:.2f}%) saved.")


Starting SAM Training...
  batch 0: loss=2.119, ascent_grad_norm=12.364
  batch 20: loss=1.461, ascent_grad_norm=4.422
  batch 40: loss=1.437, ascent_grad_norm=4.933
  batch 60: loss=1.114, ascent_grad_norm=5.031
  batch 80: loss=0.889, ascent_grad_norm=5.355
  batch 100: loss=0.517, ascent_grad_norm=5.607
  batch 120: loss=0.268, ascent_grad_norm=8.220
  batch 140: loss=0.486, ascent_grad_norm=4.601
  batch 160: loss=0.543, ascent_grad_norm=6.918
Epoch 1/30 | Loss: 163.8788 | Mean Source Val F1: 85.83%
  → new best (F1=85.83%), checkpoint saved.
  batch 0: loss=0.462, ascent_grad_norm=3.885
  batch 20: loss=0.281, ascent_grad_norm=4.712
  batch 40: loss=0.225, ascent_grad_norm=6.519
  batch 60: loss=0.308, ascent_grad_norm=6.431
  batch 80: loss=0.303, ascent_grad_norm=5.958
  batch 100: loss=0.465, ascent_grad_norm=6.060
  batch 120: loss=0.459, ascent_grad_norm=4.839
  batch 140: loss=0.421, ascent_grad_norm=4.530
  batch 160: loss=0.160, ascent_grad_norm=3.188
Epoch 2/30 | Loss: 54

In [40]:
# --- SAM evaluation ---
sam_model.load_state_dict(torch.load('/kaggle/working/sam_model.pth'))
sam_model.to(device)
sam_mean_f1, sam_worst_f1 = report_source(sam_model, "SAM")


--- SAM: Source Validation ---
  [Photo]: Accuracy = 97.01% | Macro-F1 = 96.58%
  [Art_painting]: Accuracy = 91.95% | Macro-F1 = 91.82%
  [Cartoon]: Accuracy = 94.88% | Macro-F1 = 95.48%
  Mean Source F1: 94.62% | Worst-Source F1: 91.82%


## 6. Source-Domain Separability Diagnostic
Freeze each backbone, collect balanced features from the three source-validation
sets, split 70/30 with seed 6304, and train a multinomial logistic-regression
classifier (C=1) to predict which of the three source domains a feature came from.
Held-out accuracy is the separability score — chance is 33.3%. Lower indicates
stronger invariance across the *observed* sources (no target data involved here).


In [41]:
def get_feature_extractor(model):
    if hasattr(model, 'feature_extractor'):
        return model.feature_extractor
    return nn.Sequential(*list(model.children())[:-1])

def extract_features(model, loader):
    model.eval()
    extractor = get_feature_extractor(model)
    feats = []
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            f = extractor(images)
            f = f.view(f.size(0), -1)
            feats.append(f.cpu())
    return torch.cat(feats, dim=0).numpy()

def source_domain_separability(model, tag):
    domain_ids = {'photo': 0, 'art_painting': 1, 'cartoon': 2}
    rng = np.random.RandomState(6304)

    all_feats, all_labels = [], []
    per_domain_feats = {d: extract_features(model, source_loaders[d]['val']) for d in domain_ids}
    n = min(len(f) for f in per_domain_feats.values())  # balance across domains

    for d, did in domain_ids.items():
        feats = per_domain_feats[d]
        idx = rng.choice(len(feats), size=n, replace=False)
        all_feats.append(feats[idx])
        all_labels.append(np.full(n, did))

    X = np.concatenate(all_feats, axis=0)
    y = np.concatenate(all_labels, axis=0)

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.3, random_state=6304, stratify=y
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    clf = LogisticRegression(C=1, class_weight='balanced', max_iter=5000, random_state=6304)
    clf.fit(X_train, y_train)
    separability = clf.score(X_val, y_val) * 100

    print(f"{tag:<8} source-domain separability: {separability:.2f}%  (n={n} per domain, chance=33.3%)")
    return separability

print("--- Source-Domain Separability ---")
erm_sep = source_domain_separability(erm_model, "ERM")
dandg_sep = source_domain_separability(dandg_model, "DAN-DG")
sam_sep = source_domain_separability(sam_model, "SAM")


--- Source-Domain Separability ---
ERM      source-domain separability: 89.37%  (n=334 per domain, chance=33.3%)
DAN-DG   source-domain separability: 72.43%  (n=334 per domain, chance=33.3%)
SAM      source-domain separability: 85.71%  (n=334 per domain, chance=33.3%)


## 7. Sharpness Proxy Diagnostic
Fixed validation batch (32 examples per source domain, 96 total, seed 6304). In eval
mode, measure the increase in cross-entropy after one normalized gradient-ascent
perturbation of radius 0.05 — the same standardized local diagnostic for all three
models, not a claim about global flatness.


In [42]:
def get_fixed_sharpness_batch():
    rng_state = torch.get_rng_state()
    torch.manual_seed(6304)

    images_list, labels_list = [], []
    for domain in ['photo', 'art_painting', 'cartoon']:
        ds = source_loaders[domain]['val'].dataset
        idx = torch.randperm(len(ds))[:32]
        for i in idx:
            img, lbl = ds[i]
            images_list.append(img)
            labels_list.append(lbl)

    torch.set_rng_state(rng_state)
    return torch.stack(images_list).to(device), torch.tensor(labels_list).to(device)

sharp_images, sharp_labels = get_fixed_sharpness_batch()
print(f"Fixed sharpness batch: {sharp_images.size(0)} examples")

def sharpness_proxy(model, images, labels, rho=0.05):
    model.eval()
    criterion = nn.CrossEntropyLoss()

    model.zero_grad()
    outputs = model(images)
    loss_clean = criterion(outputs, labels)
    loss_clean.backward()

    params_with_grad = [p for p in model.parameters() if p.grad is not None]
    grad_norm = torch.norm(torch.stack([p.grad.norm(p=2) for p in params_with_grad]), p=2)
    scale = rho / (grad_norm + 1e-12)

    e_ws = []
    with torch.no_grad():
        for p in params_with_grad:
            e_w = p.grad * scale
            p.add_(e_w)
            e_ws.append(e_w)

    with torch.no_grad():
        outputs_perturbed = model(images)
        loss_perturbed = criterion(outputs_perturbed, labels)

    with torch.no_grad():
        for p, e_w in zip(params_with_grad, e_ws):
            p.sub_(e_w)

    model.zero_grad()

    delta_sharp = (loss_perturbed - loss_clean).item()
    return delta_sharp

print("--- Sharpness Proxy (Delta_sharp, lower = locally flatter) ---")
erm_sharp = sharpness_proxy(erm_model, sharp_images, sharp_labels)
dandg_sharp = sharpness_proxy(dandg_model, sharp_images, sharp_labels)
sam_sharp = sharpness_proxy(sam_model, sharp_images, sharp_labels)

print(f"ERM:    {erm_sharp:.4f}")
print(f"DAN-DG: {dandg_sharp:.4f}")
print(f"SAM:    {sam_sharp:.4f}")


Fixed sharpness batch: 96 examples
--- Sharpness Proxy (Delta_sharp, lower = locally flatter) ---
ERM:    0.4146
DAN-DG: 0.4690
SAM:    0.2059


## 8. Final Evaluation on Sketch (unseen target)
**Sketch is loaded for the first time in this section**, only now that every Task 3
training, selection, and hyperparameter decision is frozen. Do not use these results
to revise anything above.


In [43]:
target_path = os.path.join(base_dir, 'sketch')
target_ds_eval = ImageFolder(root=target_path, transform=eval_transform)
target_loader_eval = DataLoader(target_ds_eval, batch_size=32, shuffle=False)
class_names = target_ds_eval.classes
print(f"Sketch (target) loaded for final evaluation only: {len(target_ds_eval)} images")
print("Class index mapping:", {i: name for i, name in enumerate(class_names)})

def sketch_eval(model, tag):
    acc, f1 = evaluate_loader(model, target_loader_eval)
    print(f"{tag:<10} Sketch Accuracy = {acc:.2f}% | Macro-F1 = {f1:.2f}%")
    return acc, f1

print("\n--- Final Sketch Evaluation ---")
erm_target_acc, erm_target_f1 = sketch_eval(erm_model, "ERM")
dandg_target_acc, dandg_target_f1 = sketch_eval(dandg_model, "DAN-DG")
sam_target_acc, sam_target_f1 = sketch_eval(sam_model, "SAM")

print(f"\nChange vs ERM — DAN-DG: {dandg_target_acc - erm_target_acc:+.2f} pts acc, "
      f"SAM: {sam_target_acc - erm_target_acc:+.2f} pts acc")


Sketch (target) loaded for final evaluation only: 3929 images
Class index mapping: {0: 'dog', 1: 'elephant', 2: 'giraffe', 3: 'guitar', 4: 'horse', 5: 'house', 6: 'person'}

--- Final Sketch Evaluation ---
ERM        Sketch Accuracy = 72.46% | Macro-F1 = 74.41%
DAN-DG     Sketch Accuracy = 71.01% | Macro-F1 = 72.11%
SAM        Sketch Accuracy = 59.38% | Macro-F1 = 65.02%

Change vs ERM — DAN-DG: -1.45 pts acc, SAM: -13.08 pts acc


### Per-class Sketch accuracy vs ERM


In [44]:
def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)

def per_class_accuracy(labels, preds, num_classes):
    cm = confusion_matrix(labels, preds, labels=list(range(num_classes)))
    with np.errstate(divide='ignore', invalid='ignore'):
        acc = np.diag(cm) / cm.sum(axis=1)
    return acc * 100, cm

models_dict = {'ERM': erm_model, 'DAN-DG': dandg_model, 'SAM': sam_model}
results = {}
confmats = {}
for name, model in models_dict.items():
    labels, preds = get_predictions(model, target_loader_eval)
    acc, cm = per_class_accuracy(labels, preds, num_classes=len(class_names))
    results[name] = acc
    confmats[name] = cm

print("--- Per-Class Sketch Accuracy ---")
header = f"{'Class':<12}" + "".join(f"{m:>10}" for m in models_dict)
print(header)
for i, cname in enumerate(class_names):
    row = f"{cname:<12}" + "".join(f"{results[m][i]:>9.1f}%" for m in models_dict)
    print(row)

for name in ['DAN-DG', 'SAM']:
    deltas = results[name] - results['ERM']
    order = np.argsort(deltas)
    print(f"\n--- {name} vs ERM (sorted worst -> best) ---")
    for idx in order:
        cname = class_names[idx]
        print(f"  {cname:<12} {deltas[idx]:+6.1f} pts  (ERM {results['ERM'][idx]:.1f}% -> {name} {results[name][idx]:.1f}%)")


--- Per-Class Sketch Accuracy ---
Class              ERM    DAN-DG       SAM
dog              54.9%     42.5%     86.3%
elephant         96.1%     85.1%     61.5%
giraffe          67.6%     78.2%     43.7%
guitar           93.9%     86.3%     66.0%
horse            58.0%     65.9%     39.1%
house            70.0%     93.8%     78.8%
person           64.4%     65.6%     62.5%

--- DAN-DG vs ERM (sorted worst -> best) ---
  dog           -12.4 pts  (ERM 54.9% -> DAN-DG 42.5%)
  elephant      -10.9 pts  (ERM 96.1% -> DAN-DG 85.1%)
  guitar         -7.6 pts  (ERM 93.9% -> DAN-DG 86.3%)
  person         +1.2 pts  (ERM 64.4% -> DAN-DG 65.6%)
  horse          +8.0 pts  (ERM 58.0% -> DAN-DG 65.9%)
  giraffe       +10.6 pts  (ERM 67.6% -> DAN-DG 78.2%)
  house         +23.8 pts  (ERM 70.0% -> DAN-DG 93.8%)

--- SAM vs ERM (sorted worst -> best) ---
  elephant      -34.6 pts  (ERM 96.1% -> SAM 61.5%)
  guitar        -28.0 pts  (ERM 93.9% -> SAM 66.0%)
  giraffe       -23.9 pts  (ERM 67.6% -> SAM

## 9. Summary table


In [45]:
print(f"{'Method':<10}{'Mean Src F1':>13}{'Worst Src F1':>14}{'Sketch Acc':>12}{'Sketch F1':>12}{'Src Separ.':>12}{'Sharpness':>12}")
print(f"{'ERM':<10}{erm_mean_f1:>12.2f}%{erm_worst_f1:>13.2f}%{erm_target_acc:>11.2f}%{erm_target_f1:>11.2f}%{erm_sep:>11.2f}%{erm_sharp:>12.4f}")
print(f"{'DAN-DG':<10}{dandg_mean_f1:>12.2f}%{dandg_worst_f1:>13.2f}%{dandg_target_acc:>11.2f}%{dandg_target_f1:>11.2f}%{dandg_sep:>11.2f}%{dandg_sharp:>12.4f}")
print(f"{'SAM':<10}{sam_mean_f1:>12.2f}%{sam_worst_f1:>13.2f}%{sam_target_acc:>11.2f}%{sam_target_f1:>11.2f}%{sam_sep:>11.2f}%{sam_sharp:>12.4f}")


Method      Mean Src F1  Worst Src F1  Sketch Acc   Sketch F1  Src Separ.   Sharpness
ERM              93.99%        91.72%      72.46%      74.41%      89.37%      0.4146
DAN-DG           94.56%        91.94%      71.01%      72.11%      72.43%      0.4690
SAM              94.62%        91.82%      59.38%      65.02%      85.71%      0.2059


In [47]:
def top_confusion(cm, class_names, true_idx):
    row = cm[true_idx].copy()
    row[true_idx] = 0  # exclude correct predictions
    total = cm[true_idx].sum()
    if row.sum() == 0:
        return None, 0, total
    pred_idx = np.argmax(row)
    return class_names[pred_idx], row[pred_idx], total

print("--- Dominant confusions: 3 worst-hit classes for SAM (vs ERM) ---")
deltas = results['SAM'] - results['ERM']
worst_classes = np.argsort(deltas)[:3]
for idx in worst_classes:
    cname = class_names[idx]
    pred_name, count, total = top_confusion(confmats['SAM'], class_names, idx)
    print(f"  {cname}: {results['SAM'][idx]:.1f}% acc — most often predicted as "
          f"'{pred_name}' ({count}/{total} examples)")

# Also check whether "dog" is absorbing other classes broadly, not just the 3 worst
print("\n--- What SAM predicts for every class it gets wrong (all classes) ---")
for idx, cname in enumerate(class_names):
    pred_name, count, total = top_confusion(confmats['SAM'], class_names, idx)
    if pred_name is not None:
        print(f"  {cname:<10} {results['SAM'][idx]:>5.1f}% acc — top wrong-prediction: "
              f"'{pred_name}' ({count}/{total - int(results['SAM'][idx]/100*total)} of errors)")

--- Dominant confusions: 3 worst-hit classes for SAM (vs ERM) ---
  elephant: 61.5% acc — most often predicted as 'dog' (283/740 examples)
  guitar: 66.0% acc — most often predicted as 'dog' (156/608 examples)
  giraffe: 43.7% acc — most often predicted as 'dog' (312/753 examples)

--- What SAM predicts for every class it gets wrong (all classes) ---
  dog         86.3% acc — top wrong-prediction: 'elephant' (69/106 of errors)
  elephant    61.5% acc — top wrong-prediction: 'dog' (283/285 of errors)
  giraffe     43.7% acc — top wrong-prediction: 'dog' (312/424 of errors)
  guitar      66.0% acc — top wrong-prediction: 'dog' (156/207 of errors)
  horse       39.1% acc — top wrong-prediction: 'dog' (432/497 of errors)
  house       78.8% acc — top wrong-prediction: 'elephant' (5/17 of errors)
  person      62.5% acc — top wrong-prediction: 'dog' (53/60 of errors)


## 10. Controlled Design Study — DAN-DG λ_DG Sweep
Hypothesis: increasing λ_DG should push the backbone toward stronger pairwise
source-domain alignment, so we'd expect source-domain separability to fall further
(more invariance across the observed sources) at some cost to source accuracy. Given
what Section 8 already showed — DAN-DG's λ=1 run cut separability substantially
(89.37% → 72.43%) without improving Sketch generalization — we don't expect a lower-
separability point here to translate into better Sketch accuracy either; the main
question is whether pushing harder makes source accuracy worse without buying
anything back on the unseen domain, echoing Task 2's λ_MMD sweep result.
λ=1 reuses the existing DAN-DG run above rather than retraining it.

In [48]:
lambda_dg_values_to_train = [0.1, 10]
dandg_sweep_models = {}

dandg_sweep_results = {
    1: {'mean_f1': dandg_mean_f1, 'worst_f1': dandg_worst_f1,
        'separability': dandg_sep, 'target_acc': dandg_target_acc, 'target_f1': dandg_target_f1}
}

for lam in lambda_dg_values_to_train:
    print(f"\n=== Training DAN-DG with lambda_dg={lam} ===")
    set_seed(6304)
    model = DANDG_ResNet()
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4, eps=1e-4)
    class_criterion = nn.CrossEntropyLoss()

    max_epochs = 30
    patience = 5
    best_mean_f1 = -1
    epochs_no_improve = 0
    best_state = None

    photo_loader = source_loaders['photo']['train']
    art_loader = source_loaders['art_painting']['train']
    cartoon_loader = source_loaders['cartoon']['train']

    for epoch in range(max_epochs):
        model.train()
        model.apply(freeze_bn_stats)
        total_class_loss, total_mmd_loss = 0, 0

        for batch_idx, ((img_p, lbl_p), (img_a, lbl_a), (img_c, lbl_c)) in enumerate(
            zip(photo_loader, art_loader, cartoon_loader)):

            img_p, lbl_p = img_p.to(device), lbl_p.to(device)
            img_a, lbl_a = img_a.to(device), lbl_a.to(device)
            img_c, lbl_c = img_c.to(device), lbl_c.to(device)

            preds_p, feats_p = model(img_p, return_features=True)
            preds_a, feats_a = model(img_a, return_features=True)
            preds_c, feats_c = model(img_c, return_features=True)

            all_preds = torch.cat([preds_p, preds_a, preds_c], dim=0)
            all_labels = torch.cat([lbl_p, lbl_a, lbl_c], dim=0)
            loss_class = class_criterion(all_preds, all_labels)

            mmd_pa = compute_mmd(feats_p, feats_a)
            mmd_pc = compute_mmd(feats_p, feats_c)
            mmd_ac = compute_mmd(feats_a, feats_c)
            loss_mmd = (mmd_pa + mmd_pc + mmd_ac) / 3.0

            total_loss = loss_class + lam * loss_mmd

            optimizer.zero_grad()
            total_loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if not torch.isfinite(grad_norm):
                optimizer.zero_grad()
                continue
            optimizer.step()

            total_class_loss += loss_class.item()
            total_mmd_loss += loss_mmd.item()

        model.eval()
        f1s = [evaluate_loader(model, source_loaders[d]['val'])[1] for d in ['photo', 'art_painting', 'cartoon']]
        mean_f1 = np.mean(f1s)
        worst_f1 = np.min(f1s)
        print(f"  Epoch {epoch+1}/{max_epochs} | Class Loss: {total_class_loss:.4f} | "
              f"MMD Loss: {total_mmd_loss:.4f} | Mean Source Val F1: {mean_f1:.2f}%")

        if mean_f1 > best_mean_f1:
            best_mean_f1 = mean_f1
            best_worst_f1 = worst_f1
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            torch.save(best_state, f'/kaggle/working/dandg_lambda_{lam}.pth')
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stopping at epoch {epoch+1} (best mean F1: {best_mean_f1:.2f}%)")
                break

    model.load_state_dict(best_state)
    dandg_sweep_models[lam] = model

    sep = source_domain_separability(model, f"DAN-DG(lambda={lam})")
    target_acc, target_f1 = sketch_eval(model, f"DAN-DG(lambda={lam})")

    dandg_sweep_results[lam] = {
        'mean_f1': best_mean_f1, 'worst_f1': best_worst_f1,
        'separability': sep, 'target_acc': target_acc, 'target_f1': target_f1
    }

print("\n--- DAN-DG Controlled Design Study: lambda_dg sweep ---")
print(f"{'Lambda':<10}{'Mean Src F1':>13}{'Worst Src F1':>14}{'Separability':>14}{'Sketch Acc':>12}{'Sketch F1':>12}")
for lam in [0.1, 1, 10]:
    r = dandg_sweep_results[lam]
    print(f"{lam:<10}{r['mean_f1']:>12.2f}%{r['worst_f1']:>13.2f}%{r['separability']:>13.2f}%{r['target_acc']:>11.2f}%{r['target_f1']:>11.2f}%")


=== Training DAN-DG with lambda_dg=0.1 ===
  Epoch 1/30 | Class Loss: 92.4631 | MMD Loss: 104.7247 | Mean Source Val F1: 90.00%
  Epoch 2/30 | Class Loss: 41.6205 | MMD Loss: 89.7294 | Mean Source Val F1: 88.89%
  Epoch 3/30 | Class Loss: 32.9825 | MMD Loss: 87.4463 | Mean Source Val F1: 89.85%
  Epoch 4/30 | Class Loss: 27.6055 | MMD Loss: 84.9551 | Mean Source Val F1: 90.92%
  Epoch 5/30 | Class Loss: 19.7965 | MMD Loss: 83.9261 | Mean Source Val F1: 93.36%
  Epoch 6/30 | Class Loss: 18.8838 | MMD Loss: 83.4807 | Mean Source Val F1: 93.62%
  Epoch 7/30 | Class Loss: 15.9909 | MMD Loss: 81.5628 | Mean Source Val F1: 93.11%
  Epoch 8/30 | Class Loss: 16.1031 | MMD Loss: 80.1188 | Mean Source Val F1: 94.02%
  Epoch 9/30 | Class Loss: 13.5040 | MMD Loss: 82.8446 | Mean Source Val F1: 92.61%
  Epoch 10/30 | Class Loss: 11.4399 | MMD Loss: 82.0563 | Mean Source Val F1: 92.96%
  Epoch 11/30 | Class Loss: 13.0805 | MMD Loss: 75.2203 | Mean Source Val F1: 92.72%
  Epoch 12/30 | Class Loss: 1